# 深度学习专题 03 · 识别数字的网络，能帮忙识别手势吗？

![一个数字 CNN 的卷积特征被复制到手势模型，三条路径展示从头训练、冻结和微调](../assets/transfer_routes.png)

给每个手势字母只留 20 张有标签的训练照片。有人提议：把已经会认手写数字的 CNN 搬过来，少学一些东西。数字 1 的竖笔与 V 手势的指缘都可能形成亮度边界，但“像数字 1”能直接变成“像字母 V”吗？

先押注：相同的手势训练预算下，从头学、只学新分类头、连同卷积层一起微调，谁会在未见手势上表现更好？把理由写在成绩表旁。本课从本地训练源模型，可独立运行。[第一课的网络和数据](CNNImageRecognition.ipynb) · [第二课的增强实验](ImageAugmentation.ipynb)

## 1 · 哪一部分可以搬走？

上一课的 CNN 可拆为两段。前半段的卷积、ReLU 和池化把一张图变为 16 张 7 × 7 特征图。后半段把 784 个特征映射成类别分数，称为**分类头**。两套图像都是单通道 28 × 28，灰度缩放也相同，可以直接复用前半段的结构和权重。

| 阶段 | 训练材料 | 分类头要回答的问题 |
| --- | --- | --- |
| 源任务预训练 | MNIST 训练子集 2,000 张 | 哪一个数字？10 个输出 |
| 目标任务适配 | 从手势训练子集中每类取 20 张，共 480 张 | 哪一个静态手势字母？24 个输出 |

运行下格让源网络真正学一遍。观察它在数字验证集上的变化。**先想**：即使数字验证成绩提高，哪一个关于手势的问题还没有得到回答？

训练图每轮刷新。Train 是本轮各批实际训练图的平均交叉熵，Validation 是轮末固定权重后在验证原图上的平均交叉熵。先观察两线是否一起下降，再结合验证准确率解释差距。

In [ ]:
# 【准备】可从项目根目录或 MachineLearning 目录运行
import matplotlib.pyplot as plt
import torch
import ipywidgets as widgets
from IPython.display import Markdown, display

try:
    from MachineLearning import image_labs as lab
except ModuleNotFoundError as exc:
    if exc.name != "MachineLearning":
        raise
    import image_labs as lab

torch.set_num_threads(min(4, torch.get_num_threads()))
device = lab.training_device()
signs = lab.load_signs()

digits = lab.load_digits()
torch.manual_seed(7)
source = lab.SmallCNN(10).to(device)
source_history = lab.fit(source, digits["train"], digits["valid"],
                         torch.optim.Adam(source.parameters(), lr=.002), epochs=6, title="MNIST / Pretraining")
print("各轮数字验证准确率", [f"{v:.1%}" for v in source_history.valid_accuracy])
target_train = lab.few_per_class(signs["train"], 20)
print("手势训练张数", len(target_train[1]))

### 用同一个核读一张数字和一张手势

两行图各展示一张验证原图和若干第一层的响应。每一列始终对应同一个数字预训练卷积核，亮处是 ReLU 后较强的正响应。比较同一列的两行：哪些亮区落在笔画或手指边缘？哪些落在背景？这给出了迁移的假设，是否能帮助分类还得用手势验证集检验。

In [ ]:
# 【观察】试着更换下标，看看刚才的判断能否在另一张照片上重复
digit_index, sign_index = 0, 0
images = torch.stack((digits["valid"][0][digit_index], signs["valid"][0][sign_index]))
with torch.no_grad():
    maps = source.features[:2](images.to(device))[:, :3].cpu()
fig, axes = plt.subplots(2, 4, figsize=(7, 3.5))
for row, label in enumerate(("Digit", "Hand")):
    axes[row, 0].imshow(images[row, 0], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(label)
    for k in range(3):
        axes[row, k+1].imshow(maps[row, k], cmap="magma", vmin=0, vmax=max(maps.max().item(), 1e-6))
        axes[row, k+1].set_title(f"Filter {k+1}")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 2 · 换头，决定哪些权重可以动

每种策略都新建一个 24 类模型。迁移组把源模型的卷积权重**复制**到目标模型，保留新初始化的手势分类头。冻结时将卷积参数的 `requires_grad` 设为 `False`，仅把分类头交给优化器。这个小网络使用卷积、ReLU 和池化，特征部分没有随训练改变的运行统计量。

| 目标策略 | 卷积起点 | 卷积学习率 | 新头学习率 |
| --- | --- | ---: | ---: |
| Scratch · 从头训练 | 随机初始化 | 0.002 | 0.002 |
| Frozen · 冻结提取器 | 数字预训练 | 冻结 | 0.002 |
| Fine-tune · 微调 | 同一份数字预训练 | 0.0005 | 0.002 |

**练习**　先写出你认为 Frozen 会更新的层名，再运行检查。找到“复制权重”和“冻结参数”各自的代码行。只复制不冻结会怎样？微调时为已学特征设置较小步长，让它能适应手势，同时控制每次改动幅度。

In [ ]:
head_lr, fine_lr = .002, .0005

def prepare_target(strategy):
    torch.manual_seed(42)  # 三组得到相同的新分类头
    model = lab.SmallCNN(len(lab.LETTERS)).to(device)
    if strategy in ("Frozen", "Fine-tune"):
        model.features.load_state_dict(source.features.state_dict())
    if strategy == "Frozen":
        model.features.requires_grad_(False)
        groups = [{"params": model.head.parameters(), "lr": head_lr}]
    else:
        feature_lr = fine_lr if strategy == "Fine-tune" else head_lr
        groups = [{"params": model.features.parameters(), "lr": feature_lr},
                  {"params": model.head.parameters(), "lr": head_lr}]
    return model, torch.optim.Adam(groups)

probe, _ = prepare_target("Frozen")
assert probe.head.out_features == 24 and source.head.out_features == 10
assert all(not p.requires_grad for p in probe.features.parameters())
assert all(p.requires_grad for p in probe.head.parameters())
for name, p in probe.named_parameters():
    print(name, tuple(p.shape), "更新" if p.requires_grad else "冻结")

## 3 · 给三种策略同样多的手势练习机会

三组都训练 12 轮，共享标签、样本顺序、新头初值和批大小。为单独观察迁移策略，目标训练统一使用原图。预训练额外使用了数字数据和计算；这里对齐的是**目标任务预算**。微调和从头训练还使用了不同的特征层学习率，所以两者之差反映整套策略。若想单独检验初始化作用，可以把两组的特征学习率也调为相同，再比较验证曲线。

**先猜再查**　Frozen 的卷积权重最大变化量应该是多少？新头呢？下格在训练前后读出实际差值，小数值也会用科学计数法显示。

In [ ]:
def snapshot(module):
    return [p.detach().clone() for p in module.parameters()]

def max_change(module, before):
    return max((p.detach() - old).abs().max().item() for p, old in zip(module.parameters(), before))

strategies = ("Scratch", "Frozen", "Fine-tune")
models, histories = {}, {}
source_before = snapshot(source.features)
for strategy in strategies:
    model, optimizer = prepare_target(strategy)
    start_features = snapshot(model.features)
    start_head = snapshot(model.head)
    if strategy != "Scratch":
        assert max_change(model.features, source_before) == 0
    histories[strategy] = lab.fit(model, target_train, signs["valid"], optimizer, epochs=12, title=strategy)
    models[strategy] = model
    feature_delta = max_change(model.features, start_features)
    head_delta = max_change(model.head, start_head)
    print(f"{strategy:10} 卷积变化 {feature_delta:.3e}，分类头变化 {head_delta:.3e}")
    assert (feature_delta == 0) == (strategy == "Frozen") and head_delta > 0
assert max_change(source.features, source_before) == 0

## 4 · 哪一种策略值得带到新任务？

看验证曲线，分别读第 1 轮和最后一轮。起跑较快的组是否也有最高的终点？训练时间很短和允许充分训练时，你的选择会一样吗？在打开测试成绩前记下选择。默认用最后一轮进行最终评估。

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for name, history in histories.items():
    values = history.valid_accuracy
    ax.plot(range(1, len(values)+1), values, ".-", label=name)
    print(name, "首轮 / 末轮验证准确率", f"{values[0]:.2%} / {values[-1]:.2%}")
ax.set(xlabel="Target epoch", ylabel="Sign validation accuracy", ylim=(0, 1))
ax.legend()
plt.show()

**最终对照**　用手势测试集的同一批 960 张图片评分。观察冻结省下了多少要更新的参数，再计算微调相对从头训练的准确率差。若源域与目标域差异较大，已有特征也可能限制目标任务表现。

下格用每轮保存的权重在测试原图上回算交叉熵。两幅图的纵轴刻度相同，可以逐轮对照训练与测试损失。测试曲线用于最后复盘；选择轮数和超参数请依据验证集。 数字预训练使用 MNIST 的测试集，手势三组使用手势测试集。

In [ ]:
# 我依据验证集选择的策略和理由：______
table = ["| 策略 | 目标可训练参数 | 手势末轮验证准确率 | 手势测试准确率 |",
         "| --- | ---: | ---: | ---: |"]
for strategy, model in models.items():
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    score = lab.accuracy(model, signs["test"])
    table.append(f"| {strategy} | {count:,} | {histories[strategy].valid_accuracy[-1]:.2%} | {score:.2%} |")
display(Markdown("\n".join(table)))

lab.plot_final_losses({"Pretraining": (source, source_history)}, digits["test"], "MNIST")
compared = {name: (model, histories[name]) for name, model in models.items()}
lab.plot_final_losses(compared, signs["test"], "Sign MNIST")


### 带走一份迁移决策

1. 写出从哪个源任务迁往哪个目标任务，指出一组被复制的权重，以及分类头重新初始化的原因。
2. 用验证曲线的一个读数、测试表的一个差值和卷积权重的实测变化，说明你是否保留开头的选择。
3. 如果把每类标注从 20 张增至 80 张，你预计哪种策略的优势会变小或变大？先给理由，再只改 `few_per_class` 的张数，重训三组，用验证集检验预测。

**延伸思考**　常见的图像迁移还会使用在大量自然图像上预训练的网络。换成这种源模型时，需要核对输入通道、图像尺寸、归一化方式和输出类别。本课的数字与手势共享输入格式，但它们的背景和纹理仍有明显差别。